# Assignment 1

In [ ]:
from pathlib import Path
import json
import re
import html
from datetime import datetime

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from bs4 import BeautifulSoup
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import RobustScaler

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)


## Extract webpage data given the url

Create a Python script that performs basic web scraping on a page to extract all the information into text and returns it as a string. String should not contain tags.

In [ ]:
def parse_web_page(url):
    """Fetch a webpage and return readable text without HTML tags."""
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; IDA-Labs-Student/1.0)"},
        timeout=30,
    )
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    # Remove content that is not part of the readable page text.
    for element in soup(["script", "style", "noscript", "svg"]):
        element.decompose()

    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", html.unescape(text)).strip()


print(parse_web_page('https://fmi.chnu.edu.ua/')[:255])
print(parse_web_page('https://en.wikipedia.org/wiki/Web_scraping')[:255])

## Get data from the API

Create a python script that performs basic request to API endpoint and saves that data to a JSON file result.json.

In [ ]:
def parse_api(api_url):
    """Fetch a JSON API response and save it to result.json."""
    response = requests.get(api_url, headers={"User-Agent": "IDA-Labs-Student/1.0"}, timeout=30)
    response.raise_for_status()
    data = response.json()
    output_path = Path("result.json")
    output_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return None


parse_api('https://api.github.com/')

## Parse the json file

Parse the `weather.json` file and return weather data for a specific date, that is given as a parameter. Return the data as an array.

In [ ]:
def parse_json(date):
    """Return all weather.json records matching date (date is normalized)."""
    path = DATA_DIR / "weather.json"
    with path.open(encoding="utf-8") as file:
        data = json.load(file)

    records = data if isinstance(data, list) else data.get("weather", data.get("data", []))
    wanted = pd.to_datetime(date).date()
    result = []
    for record in records:
        record_date = record.get("date", record.get("datetime", record.get("time", "")))
        if record_date and pd.to_datetime(record_date).date() == wanted:
            result.append(list(record.values()))
    return result


target_date = '2024-08-19'
print(parse_json(target_date))

## Parse the csv file

Parse the `weather.csv` file and return weather data for a specific date, that is given as a parameter. Return the data as an array.

In [ ]:
def parse_csv(date):
    """Return CSV weather rows for date as a list of lists."""
    df = pd.read_csv(DATA_DIR / "weather.csv")
    date_column = next((c for c in df.columns if c.lower() in {"date", "datetime", "time"}), None)
    if date_column is None:
        raise ValueError("weather.csv must contain a date/datetime/time column")
    wanted = pd.to_datetime(date).date()
    mask = pd.to_datetime(df[date_column], errors="coerce").dt.date == wanted
    return df.loc[mask].values.tolist()


target_date = '1997-05-22'
print(parse_csv(target_date))

## Visualize data

Visualize the weather.csv data using matplotlib. Choose your own approach to data visualization. Save the results (as .png, .webp files etc., your choise) in this repository.

In [ ]:
def visualize_data():
    """Create and save multiple weather visualizations from weather.csv."""
    df = pd.read_csv(DATA_DIR / "weather.csv")
    date_column = next((c for c in df.columns if c.lower() in {"date", "datetime", "time"}), None)
    if date_column is None:
        raise ValueError("weather.csv must contain a date/datetime/time column")
    df[date_column] = pd.to_datetime(df[date_column], errors="coerce")
    numeric = df.select_dtypes(include="number").columns.tolist()
    if not numeric:
        raise ValueError("weather.csv must contain at least one numeric weather column")

    # Temperature trend
    temp_col = next((c for c in numeric if "temp" in c.lower()), numeric[0])
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df[date_column], df[temp_col], marker="o", linewidth=1.5)
    ax.set(title=f"Weather trend: {temp_col}", xlabel="Date", ylabel=temp_col)
    ax.grid(alpha=0.3)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "weather_temperature_trend.png", dpi=160)
    plt.show()
    plt.close(fig)

    # Numeric weather variables comparison
    fig, ax = plt.subplots(figsize=(9, 5))
    df[numeric].plot(kind="box", ax=ax)
    ax.set(title="Distribution of weather measurements", ylabel="Value")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "weather_measurements_boxplot.png", dpi=160)
    plt.show()
    plt.close(fig)

    # Correlation heatmap
    corr = df[numeric].corr()
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(numeric)), numeric, rotation=45, ha="right")
    ax.set_yticks(range(len(numeric)), numeric)
    ax.set_title("Weather variable correlation")
    fig.colorbar(image, ax=ax, label="Correlation")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "weather_correlation.png", dpi=160)
    plt.show()
    plt.close(fig)


visualize_data()

## Data imputation and normalization.

Download kaggle dataset with incomplete/missing data, and try data imputation techniques (mean, median, mode, knn) on that data. Normalize the data with best option for each dataset (minmax, z-score, robust). Visualize the data (choose your own approach to data visualization).

Dataset options:
- Different Types of Missing Data (by Suryaprabha19) https://www.kaggle.com/datasets/suryaprabha19/different-types-of-missing-data
- Retail Product Dataset with Missing Values (by Himel Sarder) https://www.kaggle.com/datasets/himelsarder/retail-product-dataset-with-missing-values
- Data Cleaning - Feature Imputation (by 
Mr.Machine) https://www.kaggle.com/datasets/ilayaraja07/data-cleaning-feature-imputation
- BI intro to data cleaning eda and machine learning (by Walekhwa Tambiti Leo Philip) https://www.kaggle.com/datasets/walekhwatlphilip/intro-to-data-cleaning-eda-and-machine-learning

In [ ]:
def load_dataset(url, csv_file):
    """Load a Kaggle CSV with kagglehub; use the bundled fallback if unavailable."""
    try:
        import kagglehub
        from kagglehub import KaggleDatasetAdapter
        return kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, url, csv_file)
    except Exception as exc:
        fallback = DATA_DIR / "Students_Performance_knn.csv"
        if fallback.exists():
            print(f"Kaggle download unavailable ({exc}); using bundled sample dataset.")
            return pd.read_csv(fallback)
        raise RuntimeError("Could not load Kaggle dataset and no local fallback exists") from exc


def _prepare_missing_values(df):
    """Standardize common missing-value markers without modifying the input."""
    result = df.copy()
    result = result.replace(["", " ", "NA", "N/A", "na", "null", "None", "?"], np.nan)
    return result


def impute_data(df):
    """Impute numeric columns with KNN and categorical columns with mode."""
    result = _prepare_missing_values(df)
    numeric_columns = result.select_dtypes(include=np.number).columns.tolist()
    categorical_columns = result.columns.difference(numeric_columns).tolist()

    if numeric_columns:
        # KNN needs at least one observed value in each feature.
        usable_numeric = [c for c in numeric_columns if result[c].notna().any()]
        all_missing_numeric = [c for c in numeric_columns if c not in usable_numeric]
        if usable_numeric:
            imputer = KNNImputer(n_neighbors=min(5, max(1, len(result) - 1)), weights="distance")
            result[usable_numeric] = imputer.fit_transform(result[usable_numeric])
        for column in all_missing_numeric:
            result[column] = 0

    for column in categorical_columns:
        mode = result[column].mode(dropna=True)
        fill_value = mode.iloc[0] if not mode.empty else "Unknown"
        result[column] = result[column].fillna(fill_value)

    return result


def normalize_data(df):
    """Robust-scale numeric columns and keep categorical columns unchanged."""
    result = df.copy()
    numeric_columns = result.select_dtypes(include=np.number).columns.tolist()
    if numeric_columns:
        scaler = RobustScaler()
        result[numeric_columns] = scaler.fit_transform(result[numeric_columns])
    return result


def visualize_data(df):
    """Save multiple plots showing imputation/normalized dataset characteristics."""
    # Missingness by column (the input to this function may already be imputed).
    fig, ax = plt.subplots(figsize=(10, 4))
    missing = df.isna().sum().sort_values(ascending=False)
    missing.plot(kind="bar", ax=ax)
    ax.set(title="Missing values after imputation", xlabel="Column", ylabel="Missing count")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "imputation_missing_values.png", dpi=160)
    plt.show()
    plt.close(fig)

    numeric = df.select_dtypes(include=np.number)
    if not numeric.empty:
        axes = numeric.hist(figsize=(10, 6), bins=20)
        figure = axes.ravel()[0].figure
        figure.suptitle("Numeric feature distributions after robust scaling")
        figure.tight_layout()
        figure.savefig(OUTPUT_DIR / "imputation_numeric_distributions.png", dpi=160)
        plt.show()
        plt.close(figure)

        fig, ax = plt.subplots(figsize=(8, 6))
        corr = numeric.corr()
        image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
        ax.set_yticks(range(len(corr.columns)), corr.columns)
        ax.set_title("Correlation of normalized numeric features")
        fig.colorbar(image, ax=ax, label="Correlation")
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / "imputation_correlation.png", dpi=160)
        plt.show()
        plt.close(fig)


# Compare the requested imputation strategies on the selected dataset.
raw_df = load_dataset('ilayaraja07/data-cleaning-feature-imputation', 'Students_Performance_knn.csv')
raw_df = _prepare_missing_values(raw_df)

numeric_columns = raw_df.select_dtypes(include=np.number).columns.tolist()
if numeric_columns:
    comparison = pd.DataFrame(index=numeric_columns)
    comparison["missing_before"] = raw_df[numeric_columns].isna().sum()
    for strategy in ["mean", "median", "most_frequent"]:
        strategy_imputer = SimpleImputer(strategy=strategy)
        filled = strategy_imputer.fit_transform(raw_df[numeric_columns])
        comparison[strategy] = pd.isna(filled).sum(axis=0)
    knn_filled = KNNImputer(n_neighbors=5, weights="distance").fit_transform(raw_df[numeric_columns])
    comparison["knn"] = pd.isna(knn_filled).sum(axis=0)
    display(comparison)

imputed_df = impute_data(raw_df)
normalized_df = normalize_data(imputed_df)
visualize_data(normalized_df)

# Save reproducible processed data alongside the plots.
imputed_df.to_csv(DATA_DIR / "students_performance_imputed.csv", index=False)
normalized_df.to_csv(DATA_DIR / "students_performance_normalized.csv", index=False)
